# Flask

A lightweight web framework. Minimal by default — add only what you need.

**Install:** `pip install flask`  
**Run:** `python main.py` or `flask --app main run --debug`

**In this notebook:**
- Routes and HTTP methods
- Path and query parameters
- Request object (args, JSON body)
- JSON responses and status codes
- Error handlers
- Blueprints
- before_request middleware
- Application factory

> **Note:** Flask apps must be run as a server. Save code to `.py` files and run with Python or the Flask CLI.

## 1. Routes and HTTP Methods

In [ ]:
# Save as main.py and run: python main.py

from flask import Flask

app = Flask(__name__)

@app.route('/')               # GET only by default
def index():
    return 'Hello, Flask!'

@app.route('/about')
def about():
    return 'About page'

@app.route('/submit', methods=['GET', 'POST'])
def submit():
    from flask import request
    if request.method == 'POST':
        return 'Submitted!'
    return 'Show form'

if __name__ == '__main__':
    app.run(debug=True)

## 2. Path and Query Parameters

In [ ]:
from flask import Flask, request, jsonify

app = Flask(__name__)

# Path parameter — type converters: <int:>, <float:>, <string:>
@app.route('/items/<int:item_id>')
def get_item(item_id):
    return jsonify({'id': item_id})

# Query parameters — from ?key=value in the URL
@app.route('/search')
def search():
    q     = request.args.get('q', '')       # default ''
    limit = request.args.get('limit', 10, type=int)  # auto-cast
    return jsonify({'q': q, 'limit': limit, 'results': []})

# GET /items/42       → {"id": 42}
# GET /items/abc      → 404 (Flask rejects non-int)
# GET /search?q=api   → {"q": "api", "limit": 10, "results": []}

## 3. JSON Requests and Responses

In [ ]:
from flask import Flask, request, jsonify

app = Flask(__name__)

# Reading JSON body
@app.route('/users', methods=['POST'])
def create_user():
    data = request.get_json(silent=True)   # None if body is not valid JSON
    if not data or 'name' not in data:
        return jsonify({'error': 'name is required'}), 400
    return jsonify({'id': 1, 'name': data['name']}), 201

# jsonify + status code
@app.route('/items')
def list_items():
    items = [{'id': 1, 'name': 'Widget'}, {'id': 2, 'name': 'Gadget'}]
    return jsonify(items)   # 200 by default

# POST /users  Body: {"name": "Alice"}  → 201 {"id": 1, "name": "Alice"}
# POST /users  Body: {}                 → 400 {"error": "name is required"}

## 4. Error Handlers and abort

In [ ]:
from flask import Flask, request, jsonify, abort

app = Flask(__name__)

db = {1: 'Apple', 2: 'Banana'}

@app.route('/fruits/<int:fruit_id>')
def get_fruit(fruit_id):
    if fruit_id not in db:
        abort(404)   # raises exception → handled by errorhandler below
    return jsonify({'id': fruit_id, 'name': db[fruit_id]})

# Custom JSON error responses instead of HTML
@app.errorhandler(404)
def not_found(e):
    return jsonify({'error': 'Not found', 'path': request.path}), 404

@app.errorhandler(400)
def bad_request(e):
    return jsonify({'error': str(e)}), 400

# GET /fruits/1   → {"id": 1, "name": "Apple"}
# GET /fruits/99  → 404 {"error": "Not found", "path": "/fruits/99"}

## 5. Blueprints

Group related routes. Register them in the main app with a URL prefix.

In [ ]:
from flask import Flask, jsonify, Blueprint

# --- users.py ---
users_bp = Blueprint('users', __name__)

@users_bp.route('/')
def list_users():
    return jsonify([{'id': 1, 'name': 'Alice'}, {'id': 2, 'name': 'Bob'}])

@users_bp.route('/<int:user_id>')
def get_user(user_id):
    return jsonify({'id': user_id, 'name': 'Alice'})

# --- main.py ---
app = Flask(__name__)
app.register_blueprint(users_bp, url_prefix='/api/users')

# GET /api/users/      → [{"id": 1, ...}, {"id": 2, ...}]
# GET /api/users/1     → {"id": 1, "name": "Alice"}

## 6. before_request Middleware

In [ ]:
from flask import Flask, request, jsonify, g

app = Flask(__name__)

@app.before_request
def check_api_key():
    if request.path.startswith('/public'):
        return  # skip auth for public routes
    key = request.headers.get('X-API-Key')
    if not key or key != 'secret':
        return jsonify({'error': 'Unauthorized'}), 401
    g.api_key = key   # store for use in route handlers

@app.route('/public/health')
def health():
    return jsonify({'status': 'ok'})

@app.route('/private/data')
def private():
    return jsonify({'data': 'secret', 'key': g.api_key})

# GET /public/health           → 200 (no key needed)
# GET /private/data            → 401
# GET /private/data + valid key → 200

## 7. Application Factory

In [ ]:
from flask import Flask

def create_app(config=None):
    app = Flask(__name__)

    # Default config
    app.config.update({'DEBUG': False, 'TESTING': False})
    if config:
        app.config.update(config)

    # Register blueprints
    # app.register_blueprint(users_bp, url_prefix='/api/users')

    # Error handlers
    @app.errorhandler(404)
    def not_found(e):
        from flask import jsonify
        return jsonify({'error': 'Not found'}), 404

    return app

# Production
# app = create_app()

# Testing — override config
# app = create_app({'TESTING': True, 'DEBUG': True})

if __name__ == '__main__':
    app = create_app({'DEBUG': True})
    app.run(debug=True)

## Practice

| File | Difficulty | Topics |
|---|---|---|
| [01-easy.py](exercises/01-easy.py) | Easy | Routes, request, jsonify, error handlers |
| [02-medium.py](exercises/02-medium.py) | Medium | CRUD API, blueprints, abort |
| [03-challenge.py](exercises/03-challenge.py) | Challenge | App factory, auth middleware, pagination |

Solutions: [solutions/](solutions/)